In [1]:
!pip install pandas scikit-learn optuna joblib

In [2]:
import numpy as np
import pandas as pd
import joblib
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
try:
    df = pd.read_csv('/kaggle/input/itmo-pfc/dataset_downloaded.csv', index_col=0)
    print("Data loaded successfully.")
    print("Dataset size:", df.shape)
    print("First 5 raws:")
    display(df.head())
except FileNotFoundError:
    print("Error: File 'dataset.csv' not found. Make sure it is in the same directory as Notepad.")

Data loaded successfully.
Dataset size: (25478, 145)
First 5 raws:


,smiles,label,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,FpDensityMorgan1,FpDensityMorgan2,...,fr_priamide,fr_pyridine,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea
0,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,7.387216,13.399867,0.006339,-0.509118,0.583357,14.370370,383.814,1.222222,2.037037,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,6.769551,13.542332,0.041534,-0.539617,0.489619,16.500000,482.903,1.264706,2.058824,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,5.031517,4.485145,0.423142,0.423142,0.391062,11.035714,369.432,0.928571,1.785714,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)O)cc1,3.301030,11.466694,0.035411,-1.026115,0.804599,9.095238,283.287,1.142857,1.714286,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,O=C(O)/C=C/c1ccc(O)cc1,2.522879,10.103372,0.168958,-0.983264,0.650834,10.333333,164.160,1.250000,1.833333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
X = df.drop(columns=['smiles', 'label'])
y = df['label']

print("Features (X) are separated. Number of features:", X.shape[1])
print("The target variable (y) is separated.")

Features (X) are separated. Number of features: 143
The target variable (y) is separated.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training sample size: {X_train.shape[0]}")
print(f"Test sample size: {X_test.shape[0]}")

Training sample size: 20382
Test sample size: 5096


In [6]:
def objective(trial):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    layers = []
    for i in range(n_layers):
        layers.append(trial.suggest_categorical(f'n_units_l{i}', [32, 64, 128, 256]))

    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    
    solver = trial.suggest_categorical('solver', ['adam', 'sgd'])
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-3, log=True)
    alpha = trial.suggest_float('alpha', 1e-5, 1e-1, log=True)

    mlp_params = {
        'hidden_layer_sizes': tuple(layers),
        'solver': solver,
        'batch_size': batch_size,
        'learning_rate_init': learning_rate,
        'alpha': alpha,
        'max_iter': 1000,
        'early_stopping': True,
        'validation_fraction': 0.15,
        'random_state': 42,
        'n_iter_no_change': 15
    }
    
    if solver == 'sgd':
        mlp_params['momentum'] = trial.suggest_float('momentum', 0.8, 0.99)
        mlp_params['learning_rate'] = 'adaptive' 
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(**mlp_params))
    ])

    pipeline.fit(X_train, y_train)

    test_preds = pipeline.predict(X_test)
    test_r2 = r2_score(y_test, test_preds)
    
    return test_r2

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("\nOptimization complete!")
print("Best R^2:", study.best_value)
print("Best parameters:", study.best_params)

best_params = study.best_params.copy()

n_layers = best_params.pop('n_layers')
hidden_layer_sizes = tuple(best_params.pop(f'n_units_l{i}') for i in range(n_layers))

best_params['learning_rate_init'] = best_params.pop('learning_rate')

final_mlp_params = {
    'hidden_layer_sizes': hidden_layer_sizes,
    'random_state': 42,
    'max_iter': 1000,
    **best_params
}

final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(**final_mlp_params))
])

final_pipeline.fit(X_train, y_train)
print("\nThe final model was trained with the best parameters.")

[I 2025-09-26 10:46:34,088] A new study created in memory with name: no-name-b486c050-55e4-407a-9a1b-cb518860dd11
[I 2025-09-26 10:49:13,465] Trial 0 finished with value: 0.34118900665856366 and parameters: {'n_layers': 1, 'n_units_l0': 256, 'activation': 'relu', 'solver': 'adam', 'batch_size': 64, 'learning_rate': 1.2807997494281049e-05, 'alpha': 0.0007869530353864674}. Best is trial 0 with value: 0.34118900665856366.
[I 2025-09-26 10:50:04,279] Trial 1 finished with value: 0.311922880165613 and parameters: {'n_layers': 1, 'n_units_l0': 64, 'activation': 'relu', 'solver': 'sgd', 'batch_size': 128, 'learning_rate': 1.6023889528440777e-05, 'alpha': 0.03289918917076873, 'momentum': 0.9763695374552706}. Best is trial 0 with value: 0.34118900665856366.
[I 2025-09-26 10:50:30,054] Trial 2 finished with value: 0.30150345131116385 and parameters: {'n_layers': 3, 'n_units_l0': 32, 'n_units_l1': 256, 'n_units_l2': 64, 'activation': 'tanh', 'solver': 'adam', 'batch_size': 128, 'learning_rate': 0


Optimization complete!
Best R^2: 0.34717904850042525
Best parameters: {'n_layers': 1, 'n_units_l0': 128, 'activation': 'tanh', 'solver': 'sgd', 'batch_size': 64, 'learning_rate': 0.004949254291166857, 'alpha': 0.07417201845162397, 'momentum': 0.830161795808852}

The final model was trained with the best parameters.


In [7]:
y_pred = final_pipeline.predict(X_test)
r2_final = r2_score(y_test, y_pred)

print(f"The final coefficient of determination R^2 on the test set: {r2_final:.4f}")

The final coefficient of determination R^2 on the test set: 0.1737


In [8]:
model_filename = 'pIC50_mlp_regressor.joblib'
joblib.dump(final_pipeline, model_filename)

print(f"The model has been successfully saved to file: {model_filename}")

The model has been successfully saved to file: pIC50_mlp_regressor.joblib
